# Fine-tuning NLLB FR↔WO avec MLflow
Entraînement du modèle facebook/nllb-200-distilled-600M sur dataset Wolof-Français.
Tous les métriques, paramètres et le modèle sont loggés dans MLflow.

**Environnement** : Colab GPU (P100 recommandé)

## 0. Installation et configuration

In [1]:
!pip install -q \
    transformers datasets evaluate accelerate peft bitsandbytes \
    sacrebleu mlflow

import torch
print("CUDA disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 124.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
torch.cuda.empty_cache()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports et configuration MLflow

In [2]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np
import torch
import os
from datetime import datetime
import json

# MLflow
import mlflow
import mlflow.transformers

# Configuration MLflow
# Option 1 : local (Kaggle) 
# ml_ta_wo_fr
MLFLOW_TRACKING_URI = "file:///content/drive/MyDrive/mlruns"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("nllb-fr-wo-translation")

print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"MLflow Experiment: {mlflow.get_experiment_by_name('nllb-fr-wo-translation').experiment_id}")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/02/18 17:42:15 INFO mlflow.tracking.fluent: Experiment with name 'nllb-fr-wo-translation' does not exist. Creating a new experiment.


MLflow Tracking URI: file:///kaggle/working/mlruns
MLflow Experiment: 177973468392789857


## 2. Chargement et préparation du dataset

In [3]:
# Chargement
dataset_name = "galsenai/centralized_wolof_french_translation_data"
raw_ds = load_dataset(dataset_name)["train"]
raw_ds = raw_ds.rename_columns({"fr": "french", "wo": "wolof"})

print(f"Dataset brut: {len(raw_ds)} exemples")

# Doublement bidirectionnel (FR→WO + WO→FR)
def double_examples(examples):
    sources, targets, tgt_langs = [], [], []
    for fr, wo in zip(examples["french"], examples["wolof"]):
        # FR → WO
        sources.append(fr)
        targets.append(wo)
        tgt_langs.append("wol_Latn")
        # WO → FR
        sources.append(wo)
        targets.append(fr)
        tgt_langs.append("fra_Latn")
    return {"source": sources, "target": targets, "tgt_lang_code": tgt_langs}

bidir_ds = raw_ds.map(
    double_examples,
    batched=True,
    batch_size=2000,
    remove_columns=["french", "wolof", "source"],
    desc="Doublement bidirectionnel"
).shuffle(seed=42)

print(f"Dataset doublé: {len(bidir_ds)} exemples")

# Split 88 / 6 / 6
split = bidir_ds.train_test_split(test_size=0.12, seed=42)
valid_test = split["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": split["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"]
})

print(f"\nSplit final:")
print(f"  Train: {len(dataset['train'])}")
print(f"  Val:   {len(dataset['validation'])}")
print(f"  Test:  {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.81M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/98345 [00:00<?, ? examples/s]

Dataset brut: 98345 exemples


Doublement bidirectionnel:   0%|          | 0/98345 [00:00<?, ? examples/s]

Dataset doublé: 196690 exemples

Split final:
  Train: 173087
  Val:   11801
  Test:  11802


## 3. Tokenisation (NLLB spécifique)

In [4]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 128

print(f"Chargement tokenizer {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def preprocess_function(examples, tokenizer, max_length=128):
    """Tokenisation avec gestion dynamique des codes langue NLLB"""
    inputs = [str(x).strip() if x else "" for x in examples["source"]]
    targets = [str(x).strip() if x else "" for x in examples["target"]]
    
    # Tokeniser inputs
    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding=False,
    )
    
    # Tokeniser targets avec codes langue NLLB
    tgt_codes = examples["tgt_lang_code"]
    labels_tokenized = tokenizer(
        targets,
        max_length=max_length,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )
    
    # Ajouter le token de langue au début de chaque label
    labels_list = []
    for label_ids, tgt_code in zip(labels_tokenized["input_ids"], tgt_codes):
        lang_id = tokenizer.convert_tokens_to_ids(tgt_code)
        new_labels = [lang_id] + list(label_ids) + [tokenizer.eos_token_id]
        labels_list.append(new_labels)
    
    model_inputs["labels"] = labels_list
    return model_inputs

print(f"Tokenisation en cours...")
tokenized_dataset = dataset.map(
    lambda ex: preprocess_function(ex, tokenizer, max_length=MAX_LENGTH),
    batched=True,
    batch_size=1000,
    remove_columns=["source", "target", "tgt_lang_code"],
    desc="Tokenisation NLLB"
)

print(f"✓ Tokenisation terminée")

Chargement tokenizer facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Tokenisation en cours...


Tokenisation NLLB:   0%|          | 0/173087 [00:00<?, ? examples/s]

Tokenisation NLLB:   0%|          | 0/11801 [00:00<?, ? examples/s]

Tokenisation NLLB:   0%|          | 0/11802 [00:00<?, ? examples/s]

✓ Tokenisation terminée


## 4. Configuration du modèle avec LoRA

In [5]:
print(f"Chargement modèle {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Configuration LoRA pour NLLB
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Paramètres à logger
training_params = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": "q_proj,v_proj",
    "num_epochs": 4,
    "per_device_train_batch_size": 12,
    "per_device_eval_batch_size": 12,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1.2e-4,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "generation_max_length": 128,
    "generation_num_beams": 4,
    "dataset_size_train": len(tokenized_dataset["train"]),
    "dataset_size_val": len(tokenized_dataset["validation"]),
    "dataset_size_test": len(tokenized_dataset["test"]),
}

Chargement modèle facebook/nllb-200-distilled-600M...


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

trainable params: 2,359,296 || all params: 1,404,497,920 || trainable%: 0.1680


## 5. Définition des métriques

In [6]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    """Calcul BLEU et chrF"""
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Protection contre les IDs invalides
    vocab_size = tokenizer.vocab_size
    preds = np.clip(preds, 0, vocab_size - 1)
    
    # Décodage
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, vocab_size - 1)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Calcul métriques
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
        tokenize="13a"
    )
    
    chrf_result = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        char_order=6,
        word_order=2,
        beta=2
    )
    
    return {
        "bleu": bleu_result["score"],
        "chrf": chrf_result["score"]
    }

print("✓ Métriques configurées")

✓ Métriques configurées


## 6. Entraînement avec MLflow

In [13]:
output_dir = "/content/drive/MyDrive/nllb-fr-wo-lora"
os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.2e-4,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    report_to="none",  # On utilise MLflow directement via trainer
    push_to_hub=False,
    seed=42,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("✓ Trainer configuré")

✓ Trainer configuré


## 7. Exécution de l'entraînement avec logging MLflow

In [ ]:
# Démarrer une run MLflow
with mlflow.start_run() as run:
    run_id = run.info.run_id
    print(f"\n{'='*60}")
    print(f"Démarrage entraînement - Run ID: {run_id}")
    print(f"{'='*60}\n")
    
    # Logger les paramètres
    mlflow.log_params(training_params)
    
    # Logger la configuration training arguments
    training_args_dict = {
        f"training_{k}": str(v)
        for k, v in vars(training_args).items()
        if not k.startswith("_")
    }
    mlflow.log_params({k: str(v)[:500] for k, v in training_args_dict.items()})
    
    # Entraînement
    train_result = trainer.train()
    
    # Logger les métriques finales d'entraînement
    for key, value in train_result.metrics.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(f"train_{key}", value)
    
    # Évaluation sur le test
    print(f"\n{'='*60}")
    print("Évaluation sur le test set...")
    print(f"{'='*60}\n")
    
    test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
    
    for key, value in test_results.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(f"test_{key}", value)
            print(f"{key}: {value:.4f}")
    
    # Sauvegarder le modèle
    print(f"\nSauvegarde du modèle final...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # Logger le modèle dans MLflow
    mlflow.transformers.log_model(
        transformers_model=model,
        artifact_path="nllb-model",
        transformers_model_name=MODEL_NAME,
    )
    
    # Logger le tokenizer
    mlflow.log_artifact(os.path.join(output_dir, "tokenizer.model"), "tokenizer")
    mlflow.log_artifact(os.path.join(output_dir, "tokenizer_config.json"), "tokenizer")
    mlflow.log_artifact(os.path.join(output_dir, "special_tokens_map.json"), "tokenizer")
    
    # Logger les artefacts LoRA
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        mlflow.log_artifact(os.path.join(output_dir, "adapter_config.json"), "lora")
        mlflow.log_artifact(os.path.join(output_dir, "adapter_model.bin"), "lora")
    
    print(f"\n✓ Entraînement terminé")
    print(f"✓ Run ID MLflow: {run_id}")
    print(f"✓ Modèle sauvegardé dans {output_dir}")


Démarrage entraînement - Run ID: e312666484654e298ce35e8f4c9227a1



Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,8.471948,1.938847,15.671292,35.411974


## 8. Enregistrement du modèle dans le Registry MLflow (optionnel)

In [ ]:
# Optionnel: enregistrer le meilleur modèle dans le registry
run = mlflow.get_run(run_id)
model_uri = f"runs://{run_id}/nllb-model"

try:
    mlflow.register_model(
        model_uri=model_uri,
        name="nllb-fr-wo-translation",
        tags={
            "task": "machine_translation",
            "source_lang": "french",
            "target_lang": "wolof",
            "bidirectional": "true",
            "framework": "transformers",
            "adapter": "lora"
        }
    )
    print(f"✓ Modèle enregistré dans le Registry MLflow")
except Exception as e:
    print(f"Note: {e}")

print(f"\n{'='*60}")
print(f"FIN DU TRAINING - RÉSUMÉ")
print(f"{'='*60}")
print(f"BLEU Score (test): {test_results.get('eval_bleu', 'N/A')}")
print(f"chrF Score (test): {test_results.get('eval_chrf', 'N/A')}")
print(f"MLflow Run ID: {run_id}")
print(f"Model URI: {model_uri}")
print(f"{'='*60}")